# 09c — Sector/portfolio carbon intensity (data-blocked notebook)

This notebook covers portfolio- and sector-level carbon-intensity
metrics. Two of the three planned analyses depend on datasets that
aren't available here:

- A company-level carbon-intensity panel (Trucost-style data) needed
  for a sector carbon-intensity comparison — not available, so this
  analysis is omitted rather than built with placeholder data.
- An MSCI World Net-Zero-tilted index constituent panel with
  Trucost/MSCI/CDP emissions fields, needed for a portfolio-level
  intensity comparison against that index — likewise not available and
  omitted.

Only the fully self-contained worked example below, which needs no
external data, is built.


In [1]:
import numpy as np
import pandas as pd

## 1. Portfolio carbon intensity: dollar-weighted vs. naive-weighted blending

Two companies with emissions `CE`, revenue `Y` (the carbon-intensity
denominator), and market value `MV`. Each company's own carbon
intensity is `CI = CE / Y`. The two companies are then blended into a
hypothetical portfolio of size `W` at several allocation splits
`(x1, x2)`, and the portfolio carbon intensity is computed two different
ways:

- `CI_x` — the "correct" (WACI-style) approach: aggregate dollar-scaled
  emissions and dollar-scaled revenue exposure separately across the
  portfolio, then take the ratio.
- `CI2_x` — a naive approach: just take the portfolio-weighted average
  of each company's own carbon intensity.

The gap between the two demonstrates why carbon-intensity metrics don't
aggregate linearly across a portfolio the way emissions themselves do.


In [2]:
CE = np.array([5e6, 5e7])
Y = np.array([2e5, 4e6])
MV = np.array([1e7, 1e7])

CE1, CE2 = CE
Y1, Y2 = Y
MV1, MV2 = MV

CI = CE / Y
CI1, CI2 = CI
print("Company-level carbon intensity CI = CE / Y:", CI)

Company-level carbon intensity CI = CE / Y: [25.  12.5]


In [3]:
def blend(W, x1):
    x1 = np.atleast_1d(x1).astype(float)
    x2 = 1 - x1
    CE_x = W * (x1 * CE1 / MV1 + x2 * CE2 / MV2)
    Y_x = W * (x1 * Y1 / MV1 + x2 * Y2 / MV2)
    CI_x = CE_x / Y_x
    CI2_x = x1 * CI1 + x2 * CI2
    return pd.DataFrame({
        "Weight co. 1 (%)": 100 * x1,
        "Weight co. 2 (%)": 100 * x2,
        "CE_x (millions)": CE_x / 1e6,
        "Y_x (millions)": Y_x / 1e6,
        "CI_x (WACI-style)": CI_x,
        "CI2_x (naive weighted avg)": CI2_x,
    })

x1_grid = [0.00, 0.10, 0.20, 0.30, 0.50, 0.70, 0.80, 0.90, 1.00]
result1 = blend(W=1e7, x1=x1_grid).round(2)
result1

,Weight co. 1 (%),Weight co. 2 (%),CE_x (millions),Y_x (millions),CI_x (WACI-style),CI2_x (naive weighted avg)
0,0.0,100.0,50.0,4.00,12.50,12.50
1,10.0,90.0,45.5,3.62,12.57,13.75
2,20.0,80.0,41.0,3.24,12.65,15.00
3,30.0,70.0,36.5,2.86,12.76,16.25
4,50.0,50.0,27.5,2.10,13.10,18.75
5,70.0,30.0,18.5,1.34,13.81,21.25
6,80.0,20.0,14.0,0.96,14.58,22.50
7,90.0,10.0,9.5,0.58,16.38,23.75
8,100.0,0.0,5.0,0.20,25.00,25.00


In [4]:
result2 = blend(W=2e7, x1=0.5).round(2)
result2

,Weight co. 1 (%),Weight co. 2 (%),CE_x (millions),Y_x (millions),CI_x (WACI-style),CI2_x (naive weighted avg)
0,50.0,50.0,55.0,4.2,13.1,18.75
